# Mini Project 2: Survey Theme Summarizer for Student Transportation Research

This notebook builds a small research tool that helps student UX researchers summarize open-ended survey responses into themes, frequency counts, and example quotes.

In [60]:
import os
import pandas as pd

# Set up folders for the chart and generated output files.
os.makedirs("charts", exist_ok=True)
os.makedirs("output", exist_ok=True)

## Load Survey Data

The input data is a CSV of student transportation survey responses. It includes both structured survey answers and open-ended short-answer responses.

In [61]:
# Load the student transportation survey data.
df = pd.read_csv("data/student_transportation_survey.csv")

df.head()

,Timestamp,How old are you?,What's your gender?,What's your occupation?,Which area do you live in around Seattle?,Do you own a car?,How would you rate your daily commute experience using public transportation? \n(1 = Very poor 5 = Excellent),Why did you give this rating?\n(Short answer),Have you ever ridden shared e-scooters or e-bikes?,Which have you used?,When do you usually use it? (Short answer),Why do you use it? (Please briefly decribe the situation),How would you rate your overall riding experience?\n(1 = Very poor 5 = Excellent),Why did you give this rating?(Short answer),Are you planning to continue using e-scooters / e-bikes?,Why are you continue using them? (Short answer),Pick up top 3 areas you think need improvement:,Why are you not going to use them? (Short answer),How do you usually commute? (Short answer),Why have you not used shared e-scooters or e-bikes? (Short answer)
0,10/15/2025 13:57:20,18 - 24,Female,Student,uvill,No,4,Buses are convenient but can be off schedule s...,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bus,I don’t feel safe on it
1,10/15/2025 13:59:54,18 - 24,Male,Part-time,Northgate,Yes,2,"The bus is always late, the scooters don’t wor...",Yes,Both,When I need to go somewhere fast,I need to go somewhere fast but can’t drive,1.0,New York e-bikes r way better,No,NaN,"Parking experience, Payment / pricing, Limited...",Too expensive for the experience even with the...,NaN,NaN
2,10/15/2025 14:02:46,25 -34,Male,Full-time employee,Ballard,Yes,3,"Decent coverage, lacks mobile payment via appl...",Yes,Both,When I lived in Cap Hill and need to commute t...,Last Mile Commute,3.0,"It's fine, road safety in inclement weather is...",No,NaN,"Vehicle maintenance / condition, Payment / pri...",I have a car now,NaN,NaN
3,10/15/2025 14:02:53,18 - 24,Female,Student,University District,No,4,because the system is generally well-designed ...,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bus in ud! and link to downtown,never learned how to ride a bike lol
4,10/15/2025 14:03:58,18 - 24,Male,Student,UD,Yes,4,It is Ok,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,walk,not familiar with it. I need to process some a...


In [62]:
df.shape

(53, 20)

In [63]:
df.columns

Index(['Timestamp', 'How old are you?', 'What's your gender?',
       'What's your occupation?', 'Which area do you live in around Seattle?',
       'Do you own a car?',
       'How would you rate your daily commute experience using public transportation? \n(1 = Very poor 5 = Excellent) ',
       'Why did you give this rating?\n(Short answer)',
       'Have you ever ridden shared e-scooters or e-bikes?  ',
       'Which have you used?  ', 'When do you usually use it? (Short answer)',
       'Why do you use it? (Please briefly decribe the situation)',
       'How would you rate your overall riding experience?\n(1 = Very poor 5 = Excellent)  ',
       'Why did you give this rating?(Short answer)',
       'Are you planning to continue using e-scooters / e-bikes?  ',
       'Why are you continue using them? (Short answer)',
       'Pick up top 3 areas you think need improvement:  ',
       'Why are you not going to use them? (Short answer)',
       'How do you usually commute? (Short answe

## Clean Column Names

The original survey column names are long because they come directly from the survey questions. I rename the columns to shorter names so the analysis code is easier to read and maintain.

In [64]:
# Shorten the original survey question columns so the code is easier to read.
df.columns = [
    "timestamp",
    "age",
    "gender",
    "occupation",
    "area",
    "owns_car",
    "public_transit_rating",
    "public_transit_rating_reason",
    "has_used_scooter_or_ebike",
    "vehicle_type_used",
    "use_situation",
    "use_reason",
    "riding_experience_rating",
    "riding_experience_reason",
    "continue_using",
    "continue_use_reason",
    "improvement_areas",
    "stop_use_reason",
    "usual_commute",
    "non_user_reason"
]

df.columns

Index(['timestamp', 'age', 'gender', 'occupation', 'area', 'owns_car',
       'public_transit_rating', 'public_transit_rating_reason',
       'has_used_scooter_or_ebike', 'vehicle_type_used', 'use_situation',
       'use_reason', 'riding_experience_rating', 'riding_experience_reason',
       'continue_using', 'continue_use_reason', 'improvement_areas',
       'stop_use_reason', 'usual_commute', 'non_user_reason'],
      dtype='object')

## Extract Open-Ended Responses

The survey includes several short-answer columns. To analyze the responses as a research tool, I reshape these columns into a long format where each row contains one open-ended response.

In [65]:
# Keep only the short-answer columns that contain open-ended feedback.
open_response_cols = [
    "public_transit_rating_reason",
    "use_situation",
    "use_reason",
    "riding_experience_reason",
    "continue_use_reason",
    "improvement_areas",
    "stop_use_reason",
    "usual_commute",
    "non_user_reason"
]

# Turn multiple response columns into one response table.
responses_long = df[open_response_cols].reset_index().melt(
    id_vars="index",
    value_vars=open_response_cols,
    var_name="question_type",
    value_name="response_text"
)

# Rename the original row index as participant_id so each response can be traced back.
responses_long = responses_long.rename(columns={"index": "participant_id"})

responses_long.head()

,participant_id,question_type,response_text
0,0,public_transit_rating_reason,Buses are convenient but can be off schedule s...
1,1,public_transit_rating_reason,"The bus is always late, the scooters don’t wor..."
2,2,public_transit_rating_reason,"Decent coverage, lacks mobile payment via appl..."
3,3,public_transit_rating_reason,because the system is generally well-designed ...
4,4,public_transit_rating_reason,It is Ok


In [66]:
# Clean extra spaces from response text.
responses_long["response_text"] = responses_long["response_text"].astype(str).str.strip()

# Keep only rows that contain real response text.
responses_clean = responses_long[
    (responses_long["response_text"] != "") &
    (responses_long["response_text"].str.lower() != "nan")
].copy()

responses_clean.head()

,participant_id,question_type,response_text
0,0,public_transit_rating_reason,Buses are convenient but can be off schedule s...
1,1,public_transit_rating_reason,"The bus is always late, the scooters don’t wor..."
2,2,public_transit_rating_reason,"Decent coverage, lacks mobile payment via appl..."
3,3,public_transit_rating_reason,because the system is generally well-designed ...
4,4,public_transit_rating_reason,It is Ok


In [67]:
responses_clean.shape

(240, 3)

In [68]:
responses_clean["question_type"].value_counts()

question_type
public_transit_rating_reason    53
use_situation                   27
use_reason                      27
riding_experience_reason        27
improvement_areas               27
usual_commute                   26
non_user_reason                 26
continue_use_reason             15
stop_use_reason                 12
Name: count, dtype: int64

## Theme Categorization

This section assigns each open-ended response to one or more first-pass themes using keyword-based rules. This is not meant to replace full qualitative coding, but it helps create a quick summary of repeated concerns in a small student research dataset.

In [69]:
# These keywords define the first-pass themes for the survey responses.
theme_keywords = {
    "Reliability / schedule": [
        "late", "delay", "delayed", "schedule", "off schedule",
        "unreliable", "reliable", "waiting", "wait", "on time"
    ],
    "Safety concerns": [
        "unsafe", "not safe", "safety", "danger", "dangerous",
        "traffic", "helmet", "road condition", "bad roads", "weather"
    ],
    "Cost / pricing": [
        "expensive", "price", "pricing", "cost", "fee",
        "charge", "cheap", "affordable", "pass"
    ],
    "Payment / app friction": [
        "payment", "apple pay", "mobile payment", "app",
        "card", "pay through", "pay via"
    ],
    "Availability / access": [
        "available", "availability", "access", "accessible",
        "limited", "coverage", "nearby", "not enough", "hard to find"
    ],
    "Parking / pickup / dropoff": [
        "parking", "park", "pickup", "pick up",
        "dropoff", "drop off", "dock", "drop-off", "pick-up"
    ],
    "Vehicle condition": [
        "maintenance", "condition", "broken", "battery",
        "dirty", "doesn't work", "don’t work", "not work", "malfunction"
    ],
    "Convenience / speed": [
        "convenient", "fast", "quick", "speed",
        "easy", "efficient", "last mile", "short trip"
    ],
    "Preference for other transportation": [
        "prefer bus", "prefer walking", "prefer to walk",
        "prefer driving", "prefer my car", "i drive",
        "i walk", "use my car", "take the bus instead",
        "use link instead"
    ],
    "Lack of familiarity / skill": [
        "never used", "haven't used", "have not used",
        "not familiar", "don't know how", "do not know how",
        "never learned", "learned how"
    ]
}

In [70]:
def assign_themes(text):
    # Match themes using lowercase text so capitalization does not matter.
    text_lower = str(text).lower()
    matched_themes = []

    # Check each theme's keyword list against the response.
    for theme, keywords in theme_keywords.items():
        for keyword in keywords:
            if keyword in text_lower:
                matched_themes.append(theme)
                break

    # Keep unmatched responses visible instead of forcing them into a theme.
    if matched_themes:
        return matched_themes
    else:
        return ["Other / unclear"]

In [71]:
# Assign one or more themes to each open-ended response.
responses_clean["themes"] = responses_clean["response_text"].apply(assign_themes)

responses_clean.head()

,participant_id,question_type,response_text,themes
0,0,public_transit_rating_reason,Buses are convenient but can be off schedule s...,"[Reliability / schedule, Convenience / speed]"
1,1,public_transit_rating_reason,"The bus is always late, the scooters don’t wor...","[Reliability / schedule, Cost / pricing, Parki..."
2,2,public_transit_rating_reason,"Decent coverage, lacks mobile payment via appl...","[Safety concerns, Payment / app friction, Avai..."
3,3,public_transit_rating_reason,because the system is generally well-designed ...,"[Reliability / schedule, Payment / app frictio..."
4,4,public_transit_rating_reason,It is Ok,[Other / unclear]


## Theme Frequency Summary

After assigning themes, I expand the theme list so each theme can be counted. This creates a frequency-ranked summary of recurring concerns in the survey responses.

In [72]:
# Split multi-theme responses so each theme can be counted.
theme_rows = responses_clean.explode("themes")

# Count how often each theme appears.
theme_frequency = (
    theme_rows["themes"]
    .value_counts()
    .reset_index()
)

theme_frequency.columns = ["theme", "frequency"]

theme_frequency

,theme,frequency
0,Other / unclear,115
1,Cost / pricing,38
2,Safety concerns,35
3,Convenience / speed,31
4,Payment / app friction,25
5,Availability / access,21
6,Vehicle condition,17
7,Reliability / schedule,16
8,Parking / pickup / dropoff,10
9,Preference for other transportation,10


In [73]:
# Save the frequency table as one of the tool outputs.
theme_frequency.to_csv("output/theme_frequency.csv", index=False)

## Example Responses by Theme

A frequency table shows which themes appear most often, but example responses help explain what those themes mean in the participants' own words.

In [74]:
# Pull a few example responses for each theme to make the summary easier to interpret.
example_responses = (
    theme_rows
    .groupby("themes")["response_text"]
    .apply(lambda responses: list(responses.head(3)))
    .reset_index()
)

example_responses.columns = ["theme", "example_responses"]

example_responses

,theme,example_responses
0,Availability / access,"[Decent coverage, lacks mobile payment via app..."
1,Convenience / speed,[Buses are convenient but can be off schedule ...
2,Cost / pricing,"[The bus is always late, the scooters don’t wo..."
3,Lack of familiarity / skill,"[I haven't used it since I got a car, but I us..."
4,Other / unclear,"[It is Ok, just soso, I love public transit (t..."
5,Parking / pickup / dropoff,"[The bus is always late, the scooters don’t wo..."
6,Payment / app friction,"[Decent coverage, lacks mobile payment via app..."
7,Preference for other transportation,"[I drive to campus, so I don't take the public..."
8,Reliability / schedule,[Buses are convenient but can be off schedule ...
9,Safety concerns,"[Decent coverage, lacks mobile payment via app..."


In [75]:
# Save example responses as another tool output.
example_responses.to_csv("output/example_responses_by_theme.csv", index=False)

## Theme Frequency Chart

This chart visualizes the frequency-ranked themes from the keyword-based categorization. It helps a student UX researcher quickly see which concerns appear most often in the open-ended survey responses.

In [76]:
import plotly.express as px

# Sort themes so the horizontal bar chart is easier to read.
theme_frequency_sorted = theme_frequency.sort_values("frequency", ascending=True)

fig = px.bar(
    theme_frequency_sorted,
    x="frequency",
    y="theme",
    orientation="h",
    title="Frequency of Themes in Open-Ended Transportation Survey Responses",
    labels={
        "frequency": "Number of Theme Mentions",
        "theme": "Theme"
    }
)

fig.show()

# Save the chart so it can be reviewed outside the notebook.
fig.write_image("charts/theme_frequency.png")

**Interpretation:**  
This chart shows that many responses were categorized as “Other / unclear,” which reflects a limitation of the keyword-based categorization approach. Among the more specific themes, cost/pricing, safety concerns, convenience/speed, payment/app friction, and availability/access appear most often. This suggests that the tool is useful for creating a first-pass map of recurring concerns, but the results should still be reviewed by a researcher before being treated as final findings.

## Generate Markdown Report

The final output is a short markdown report that summarizes the theme frequency results and includes a note about limitations.

In [77]:
# Keep the report focused on specific themes rather than "Other / unclear."
top_specific_themes = theme_frequency[
    theme_frequency["theme"] != "Other / unclear"
].head(5)

# Build a short markdown report from the theme frequency results.
report_lines = [
    "# Survey Theme Summary Report",
    "",
    "## Overview",
    "This report summarizes recurring themes from open-ended responses in a student transportation survey.",
    "",
    "## Top Specific Themes",
]

for _, row in top_specific_themes.iterrows():
    report_lines.append(f"- **{row['theme']}**: {row['frequency']} mentions")

report_lines.extend([
    "",
    "## Interpretation",
    "The keyword-based categorization surfaced several recurring concerns in the survey responses. Among the specific themes, cost/pricing, safety concerns, convenience/speed, payment/app friction, and availability/access appeared frequently.",
    "",
    "The high number of responses categorized as 'Other / unclear' shows that this tool should be used as a first-pass synthesis aid rather than a replacement for manual qualitative coding. A researcher should review the example responses and refine the theme rules before using the results as final findings.",
])

report_text = "\n".join(report_lines)

# Save the generated report as a markdown file.
with open("output/theme_summary_report.md", "w") as f:
    f.write(report_text)

print(report_text)

# Survey Theme Summary Report

## Overview
This report summarizes recurring themes from open-ended responses in a student transportation survey.

## Top Specific Themes
- **Cost / pricing**: 38 mentions
- **Safety concerns**: 35 mentions
- **Convenience / speed**: 31 mentions
- **Payment / app friction**: 25 mentions
- **Availability / access**: 21 mentions

## Interpretation
The keyword-based categorization surfaced several recurring concerns in the survey responses. Among the specific themes, cost/pricing, safety concerns, convenience/speed, payment/app friction, and availability/access appeared frequently.

The high number of responses categorized as 'Other / unclear' shows that this tool should be used as a first-pass synthesis aid rather than a replacement for manual qualitative coding. A researcher should review the example responses and refine the theme rules before using the results as final findings.


## Tool Outputs

This notebook produces three output files:

- `output/theme_frequency.csv`: a frequency-ranked table of assigned themes
- `output/example_responses_by_theme.csv`: example responses for each theme
- `output/theme_summary_report.md`: a short markdown report summarizing the main theme patterns and limitations

It also saves one chart:

- `charts/theme_frequency.png`